# 02 - XGBoost / LightGBM Modeling (Improved)

This notebook trains gradient boosting models for sales prediction.

**Key Improvements:**
1. **Horizon-aware feature engineering** - No lag/rolling features shorter than forecast horizon
2. **Proper temporal cross-validation** with TimeSeriesSplit
3. **LightGBM + XGBoost comparison** with ensemble option
4. **Optuna hyperparameter tuning** (optional)
5. **Multi-series support** - Group-aware features (store_id, product_id)
6. **Comprehensive metrics** - MAE, RMSE, R2, MAPE, sMAPE, bias analysis
7. **Feature importance analysis** with selection
8. **Log-transform option** for heteroscedastic data

In [ ]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import json
import joblib
import datetime
import warnings
warnings.filterwarnings('ignore')

# Project setup
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.features import build_feature_pipeline

print(f"[INFO] Project root: {PROJECT_ROOT}")

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Paths
FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "train_features.csv"
CLEAN_PATH = PROJECT_ROOT / "data" / "interim" / "train_clean.csv"  # Fallback
FORECAST_PATH = PROJECT_ROOT / "models" / "artifacts" / "xgb_forecast_final.csv"
METRICS_PATH = PROJECT_ROOT / "models" / "metrics" / "xgb_metrics_final.json"
MODEL_PATH = PROJECT_ROOT / "models" / "artifacts"

MODEL_PATH.mkdir(parents=True, exist_ok=True)
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)

# Modeling parameters
HORIZON = 14  # Forecast horizon in days (CRITICAL: features must be >= HORIZON)
N_CV_SPLITS = 5  # Number of CV folds
USE_OPTUNA = False  # Set True for hyperparameter tuning (slower)
OPTUNA_TRIALS = 30  # Number of Optuna trials
USE_LOG_TARGET = False  # Log-transform target for heteroscedastic data

# Feature engineering - IMPORTANT: All lags and windows must be >= HORIZON
# to avoid data leakage at inference time
LAGS = [l for l in [14, 21, 28, 35, 42] if l >= HORIZON]
WINDOWS = [w for w in [14, 21, 28] if w >= HORIZON]

print(f"[CONFIG] Horizon: {HORIZON} days")
print(f"[CONFIG] Lags (>= horizon): {LAGS}")
print(f"[CONFIG] Windows (>= horizon): {WINDOWS}")
print(f"[CONFIG] Use log target: {USE_LOG_TARGET}")
print(f"[CONFIG] Use Optuna tuning: {USE_OPTUNA}")

## 1. Data Loading

In [ ]:
# Try to load features file, fallback to clean file
if FEATURES_PATH.exists():
    df = pd.read_csv(FEATURES_PATH, parse_dates=['date'])
    print(f"[LOAD] Loaded features from: {FEATURES_PATH}")
elif CLEAN_PATH.exists():
    df = pd.read_csv(CLEAN_PATH, parse_dates=['date'])
    print(f"[LOAD] Loaded clean data from: {CLEAN_PATH}")
else:
    raise FileNotFoundError(f"No data file found at {FEATURES_PATH} or {CLEAN_PATH}")

print(f"[LOAD] Shape: {df.shape}")
print(f"[LOAD] Columns: {list(df.columns)}")
print(f"[LOAD] Date range: {df['date'].min()} to {df['date'].max()}")

In [ ]:
# Check for multi-series structure
group_cols = [c for c in ['store_id', 'product_id'] if c in df.columns]
print(f"[INFO] Group columns found: {group_cols}")

if group_cols:
    n_groups = df.groupby(group_cols).ngroups
    print(f"[INFO] Number of unique groups: {n_groups}")
    print(f"[INFO] Average samples per group: {len(df) / n_groups:.1f}")

# Value statistics
print(f"\n[INFO] Target 'value' statistics:")
print(df['value'].describe())

## 2. Data Preprocessing

In [ ]:
# Sort by group + date for correct temporal ordering
sort_cols = group_cols + ['date']
df = df.sort_values(sort_cols).reset_index(drop=True)

# Check for and handle duplicates
if group_cols:
    dup_cols = group_cols + ['date']
else:
    dup_cols = ['date']
    
n_duplicates = df.duplicated(subset=dup_cols).sum()
if n_duplicates > 0:
    print(f"[PREPROCESS] Found {n_duplicates} duplicate rows, keeping first occurrence")
    df = df.drop_duplicates(subset=dup_cols, keep='first')

# Handle negative values
n_negative = (df['value'] < 0).sum()
if n_negative > 0:
    print(f"[PREPROCESS] Clipping {n_negative} negative values to 0")
    df['value'] = df['value'].clip(lower=0)

# Optional: Winsorize extreme outliers (99.5th percentile)
upper_cap = df['value'].quantile(0.995)
n_capped = (df['value'] > upper_cap).sum()
if n_capped > 0:
    print(f"[PREPROCESS] Capping {n_capped} extreme values at {upper_cap:.2f}")
    df['value'] = df['value'].clip(upper=upper_cap)

print(f"[PREPROCESS] Final shape: {df.shape}")

## 3. Feature Engineering (Horizon-Aware)

In [ ]:
# Remove any pre-existing lag/rolling features that are shorter than horizon
# This is CRITICAL to avoid data leakage
import re

short_feature_pattern = re.compile(
    r'^(?:lag|roll_mean|roll_std|roll_min|roll_max|roll_range|roll_cv|ewma|lag_diff|lag_ratio)_(\d+)'
)

cols_to_drop = []
for col in df.columns:
    match = short_feature_pattern.match(col)
    if match:
        lag_val = int(match.group(1))
        if lag_val < HORIZON:
            cols_to_drop.append(col)

if cols_to_drop:
    print(f"[FEATURES] Dropping {len(cols_to_drop)} short-horizon features: {cols_to_drop[:5]}...")
    df = df.drop(columns=cols_to_drop, errors='ignore')

# Build features using the shared pipeline
print(f"[FEATURES] Building features with lags={LAGS}, windows={WINDOWS}")

df_feat, encoders = build_feature_pipeline(
    df,
    lags=LAGS,
    windows=WINDOWS,
    group_cols=group_cols if group_cols else None,
    is_train=True,
    horizon=HORIZON,
)

# Drop NaN rows from lag/rolling warm-up
n_before = len(df_feat)
df_feat = df_feat.dropna().reset_index(drop=True)
n_after = len(df_feat)
print(f"[FEATURES] Dropped {n_before - n_after} NaN rows from warm-up period")
print(f"[FEATURES] Final feature shape: {df_feat.shape}")

In [ ]:
# Prepare X and y
EXCLUDE_COLS = {'value', 'date', 'id'}
feature_cols = [c for c in df_feat.columns if c not in EXCLUDE_COLS]

X = df_feat[feature_cols].copy()
y = df_feat['value'].copy()

# Optional: Log-transform target
if USE_LOG_TARGET:
    y_orig = y.copy()
    y = np.log1p(y)
    print(f"[FEATURES] Applied log1p transform to target")

print(f"[FEATURES] X shape: {X.shape}")
print(f"[FEATURES] y shape: {y.shape}")
print(f"[FEATURES] Feature columns ({len(feature_cols)}):")
print(f"  {feature_cols[:10]}..." if len(feature_cols) > 10 else f"  {feature_cols}")

## 4. Train/Test Split (Temporal)

In [ ]:
# Get unique dates for proper temporal split
unique_dates = sorted(df_feat['date'].unique())
n_unique_dates = len(unique_dates)

# Adjust horizon if needed
actual_horizon = min(HORIZON, n_unique_dates // 5)  # Max 20% for test
if actual_horizon < HORIZON:
    print(f"[SPLIT] Adjusted horizon from {HORIZON} to {actual_horizon} (not enough dates)")

cutoff_date = unique_dates[-actual_horizon]

train_mask = df_feat['date'] < cutoff_date
test_mask = df_feat['date'] >= cutoff_date

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]
dates_test = df_feat.loc[test_mask, 'date']

if USE_LOG_TARGET:
    y_train_orig = y_orig[train_mask]
    y_test_orig = y_orig[test_mask]

print(f"[SPLIT] Cutoff date: {cutoff_date}")
print(f"[SPLIT] Train: {len(X_train)} samples ({train_mask.sum() / len(df_feat) * 100:.1f}%)")
print(f"[SPLIT] Test: {len(X_test)} samples ({test_mask.sum() / len(df_feat) * 100:.1f}%)")

## 5. Model Training

In [ ]:
# Default hyperparameters (tuned for robustness)
LGBM_PARAMS = {
    'n_estimators': 1000,
    'max_depth': 8,
    'learning_rate': 0.03,
    'num_leaves': 31,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'verbosity': -1,
    'n_jobs': -1,
}

XGB_PARAMS = {
    'n_estimators': 1000,
    'max_depth': 6,
    'learning_rate': 0.05,
    'min_child_weight': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': 42,
    'early_stopping_rounds': 50,
    'verbosity': 0,
}

In [ ]:
# Optional: Optuna hyperparameter tuning
if USE_OPTUNA:
    try:
        import optuna
        from lightgbm import LGBMRegressor
        
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        
        def objective(trial):
            params = {
                'n_estimators': 500,
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
                'num_leaves': trial.suggest_int('num_leaves', 15, 63),
                'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
                'random_state': 42,
                'verbosity': -1,
            }
            
            # TimeSeriesSplit CV
            tscv = TimeSeriesSplit(n_splits=3)
            scores = []
            
            for train_idx, val_idx in tscv.split(X_train):
                X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
                y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
                
                model = LGBMRegressor(**params)
                model.fit(X_tr, y_tr)
                y_pred = model.predict(X_val)
                
                if USE_LOG_TARGET:
                    y_pred = np.expm1(y_pred)
                    y_val_actual = np.expm1(y_val)
                else:
                    y_val_actual = y_val
                
                scores.append(mean_absolute_error(y_val_actual, y_pred))
            
            return np.mean(scores)
        
        study = optuna.create_study(direction='minimize')
        study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
        
        # Update params with best values
        LGBM_PARAMS.update(study.best_params)
        print(f"[OPTUNA] Best MAE: {study.best_value:.4f}")
        print(f"[OPTUNA] Best params: {study.best_params}")
        
    except ImportError:
        print("[WARNING] Optuna not installed, using default parameters")

In [ ]:
# Train LightGBM
try:
    from lightgbm import LGBMRegressor, early_stopping, log_evaluation
    
    model_lgbm = LGBMRegressor(**LGBM_PARAMS)
    model_lgbm.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[early_stopping(50), log_evaluation(-1)],
    )
    
    print(f"[LGBM] Best iteration: {model_lgbm.best_iteration_}")
    USE_LGBM = True
    
except ImportError:
    print("[WARNING] LightGBM not installed")
    model_lgbm = None
    USE_LGBM = False

In [ ]:
# Train XGBoost
from xgboost import XGBRegressor

model_xgb = XGBRegressor(**XGB_PARAMS)
model_xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

print(f"[XGB] Best iteration: {model_xgb.best_iteration}")

## 6. Model Evaluation

In [ ]:
def compute_all_metrics(y_true, y_pred, prefix=''):
    """Compute comprehensive metrics."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    bias = np.mean(y_pred - y_true)
    
    # MAPE (excluding near-zeros)
    mask = np.abs(y_true) > 1.0
    if mask.sum() > 0:
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    else:
        mape = None
    
    # sMAPE
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    denom = np.where(denom > 0, denom, 1.0)
    smape = np.mean(np.abs(y_true - y_pred) / denom) * 100
    
    metrics = {
        f'{prefix}MAE': mae,
        f'{prefix}RMSE': rmse,
        f'{prefix}R2': r2,
        f'{prefix}Bias': bias,
        f'{prefix}MAPE': mape,
        f'{prefix}sMAPE': smape,
    }
    return metrics


def print_metrics(metrics, title):
    """Pretty print metrics."""
    print(f"\n{'='*60}")
    print(f"{title}")
    print('='*60)
    for key, val in metrics.items():
        if val is not None:
            if 'R2' in key:
                print(f"  {key:15s}: {val:.4f}")
            elif 'MAPE' in key or 'sMAPE' in key:
                print(f"  {key:15s}: {val:.2f}%")
            else:
                print(f"  {key:15s}: {val:.4f}")
    print('='*60)

In [ ]:
# Evaluate both models
results = {}

# XGBoost evaluation
y_pred_xgb = model_xgb.predict(X_test)
y_pred_xgb_train = model_xgb.predict(X_train)

if USE_LOG_TARGET:
    y_pred_xgb = np.expm1(y_pred_xgb)
    y_pred_xgb_train = np.expm1(y_pred_xgb_train)
    y_test_eval = y_test_orig.values
    y_train_eval = y_train_orig.values
else:
    y_test_eval = y_test.values
    y_train_eval = y_train.values

metrics_xgb_train = compute_all_metrics(y_train_eval, y_pred_xgb_train, 'Train_')
metrics_xgb_test = compute_all_metrics(y_test_eval, y_pred_xgb, 'Test_')
results['XGBoost'] = {**metrics_xgb_train, **metrics_xgb_test}

print_metrics({**metrics_xgb_train, **metrics_xgb_test}, 'XGBoost Results')

# Overfitting ratio
overfit_ratio = metrics_xgb_test['Test_RMSE'] / metrics_xgb_train['Train_RMSE']
print(f"  Overfit ratio (RMSE_test/RMSE_train): {overfit_ratio:.2f}")
results['XGBoost']['Overfit_ratio'] = overfit_ratio

In [ ]:
# LightGBM evaluation (if available)
if USE_LGBM:
    y_pred_lgbm = model_lgbm.predict(X_test)
    y_pred_lgbm_train = model_lgbm.predict(X_train)
    
    if USE_LOG_TARGET:
        y_pred_lgbm = np.expm1(y_pred_lgbm)
        y_pred_lgbm_train = np.expm1(y_pred_lgbm_train)
    
    metrics_lgbm_train = compute_all_metrics(y_train_eval, y_pred_lgbm_train, 'Train_')
    metrics_lgbm_test = compute_all_metrics(y_test_eval, y_pred_lgbm, 'Test_')
    results['LightGBM'] = {**metrics_lgbm_train, **metrics_lgbm_test}
    
    print_metrics({**metrics_lgbm_train, **metrics_lgbm_test}, 'LightGBM Results')
    
    overfit_ratio_lgbm = metrics_lgbm_test['Test_RMSE'] / metrics_lgbm_train['Train_RMSE']
    print(f"  Overfit ratio (RMSE_test/RMSE_train): {overfit_ratio_lgbm:.2f}")
    results['LightGBM']['Overfit_ratio'] = overfit_ratio_lgbm

## 7. Cross-Validation

In [ ]:
# TimeSeriesSplit cross-validation
tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS)

cv_results = {'MAE': [], 'RMSE': [], 'R2': []}

print(f"[CV] Running {N_CV_SPLITS}-fold TimeSeriesSplit...")

for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Train model for this fold
    if USE_LGBM:
        model_cv = LGBMRegressor(**{**LGBM_PARAMS, 'n_estimators': 500})
        model_cv.fit(X_tr, y_tr)
    else:
        model_cv = XGBRegressor(**{**XGB_PARAMS, 'n_estimators': 500, 'early_stopping_rounds': None})
        model_cv.fit(X_tr, y_tr)
    
    y_pred_cv = model_cv.predict(X_val)
    
    if USE_LOG_TARGET:
        y_pred_cv = np.expm1(y_pred_cv)
        y_val_actual = np.expm1(y_val) if USE_LOG_TARGET else y_val
    else:
        y_val_actual = y_val
    
    mae_fold = mean_absolute_error(y_val_actual, y_pred_cv)
    rmse_fold = np.sqrt(mean_squared_error(y_val_actual, y_pred_cv))
    r2_fold = r2_score(y_val_actual, y_pred_cv)
    
    cv_results['MAE'].append(mae_fold)
    cv_results['RMSE'].append(rmse_fold)
    cv_results['R2'].append(r2_fold)
    
    print(f"  Fold {fold+1}: MAE={mae_fold:.2f}, RMSE={rmse_fold:.2f}, R2={r2_fold:.4f}")

print(f"\n[CV] Summary:")
print(f"  MAE  : {np.mean(cv_results['MAE']):.4f} (+/- {np.std(cv_results['MAE']):.4f})")
print(f"  RMSE : {np.mean(cv_results['RMSE']):.4f} (+/- {np.std(cv_results['RMSE']):.4f})")
print(f"  R2   : {np.mean(cv_results['R2']):.4f} (+/- {np.std(cv_results['R2']):.4f})")

## 8. Feature Importance

In [ ]:
# Get feature importance from best model
best_model = model_lgbm if USE_LGBM else model_xgb
model_name = 'LightGBM' if USE_LGBM else 'XGBoost'

importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

# Plot top 20 features
fig, ax = plt.subplots(figsize=(10, 8))
top_n = min(20, len(importance_df))
sns.barplot(data=importance_df.head(top_n), x='importance', y='feature', ax=ax)
ax.set_title(f'{model_name} - Top {top_n} Feature Importance')
ax.set_xlabel('Importance')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

print(f"\n[IMPORTANCE] Top 10 features:")
for idx, row in importance_df.head(10).iterrows():
    print(f"  {row['feature']:30s}: {row['importance']:.4f}")

## 9. Visualization

In [ ]:
# Plot actual vs predicted
y_pred_best = y_pred_lgbm if USE_LGBM else y_pred_xgb

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Time series comparison
axes[0, 0].plot(dates_test.values, y_test_eval, label='Actual', marker='o', alpha=0.7)
axes[0, 0].plot(dates_test.values, y_pred_best, label='Predicted', marker='s', alpha=0.7)
axes[0, 0].set_title(f'{model_name}: Actual vs Predicted (Test Set)')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Value')
axes[0, 0].legend()
axes[0, 0].tick_params(axis='x', rotation=45)

# Scatter plot
axes[0, 1].scatter(y_test_eval, y_pred_best, alpha=0.5)
max_val = max(y_test_eval.max(), y_pred_best.max())
axes[0, 1].plot([0, max_val], [0, max_val], 'r--', label='Perfect')
axes[0, 1].set_title(f'{model_name}: Scatter Plot (R2={results[model_name]["Test_R2"]:.3f})')
axes[0, 1].set_xlabel('Actual')
axes[0, 1].set_ylabel('Predicted')
axes[0, 1].legend()

# Residuals
residuals = y_test_eval - y_pred_best
axes[1, 0].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].axvline(x=0, color='r', linestyle='--')
axes[1, 0].set_title(f'{model_name}: Residual Distribution')
axes[1, 0].set_xlabel('Residual (Actual - Predicted)')
axes[1, 0].set_ylabel('Frequency')

# Residuals vs Predicted
axes[1, 1].scatter(y_pred_best, residuals, alpha=0.5)
axes[1, 1].axhline(y=0, color='r', linestyle='--')
axes[1, 1].set_title(f'{model_name}: Residuals vs Predicted')
axes[1, 1].set_xlabel('Predicted')
axes[1, 1].set_ylabel('Residual')

plt.tight_layout()
plt.show()

## 10. Save Artifacts

In [ ]:
# Prepare final metrics
best_results = results[model_name]

final_metrics = {
    'model_type': model_name,
    'horizon': HORIZON,
    'train_size': len(X_train),
    'test_size': len(X_test),
    'n_features': len(feature_cols),
    'use_log_target': USE_LOG_TARGET,
    'lags': LAGS,
    'windows': WINDOWS,
    # Train metrics
    'MAE_train': float(best_results['Train_MAE']),
    'RMSE_train': float(best_results['Train_RMSE']),
    'R2_train': float(best_results['Train_R2']),
    # Test metrics
    'MAE_test': float(best_results['Test_MAE']),
    'RMSE_test': float(best_results['Test_RMSE']),
    'R2_test': float(best_results['Test_R2']),
    'Bias_test': float(best_results['Test_Bias']),
    'MAPE_test': float(best_results['Test_MAPE']) if best_results['Test_MAPE'] else None,
    'sMAPE_test': float(best_results['Test_sMAPE']),
    'Overfit_ratio': float(best_results['Overfit_ratio']),
    # CV metrics
    'CV_MAE_mean': float(np.mean(cv_results['MAE'])),
    'CV_MAE_std': float(np.std(cv_results['MAE'])),
    'CV_R2_mean': float(np.mean(cv_results['R2'])),
    'CV_R2_std': float(np.std(cv_results['R2'])),
}

# Save metrics
with open(METRICS_PATH, 'w') as f:
    json.dump(final_metrics, f, indent=2)
print(f"[SAVE] Metrics saved: {METRICS_PATH}")

# Save model
today = datetime.datetime.now().strftime('%Y%m%d')
model_file = MODEL_PATH / f"model_{today}_{model_name.lower()}.pkl"
joblib.dump(best_model, model_file)
print(f"[SAVE] Model saved: {model_file}")

# Save encoders
encoders_file = MODEL_PATH / f"encoders_{today}.pkl"
joblib.dump(encoders, encoders_file)
print(f"[SAVE] Encoders saved: {encoders_file}")

# Save forecast
forecast_df = pd.DataFrame({
    'date': dates_test.values,
    'actual': y_test_eval,
    'predicted': y_pred_best,
    'residual': y_test_eval - y_pred_best,
})
forecast_df.to_csv(FORECAST_PATH, index=False)
print(f"[SAVE] Forecast saved: {FORECAST_PATH}")

In [ ]:
# Final summary
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"  Model        : {model_name}")
print(f"  Horizon      : {HORIZON} days")
print(f"  Features     : {len(feature_cols)}")
print(f"  Train samples: {len(X_train)}")
print(f"  Test samples : {len(X_test)}")
print(f"\n  Test Metrics:")
print(f"    MAE        : {final_metrics['MAE_test']:.4f}")
print(f"    RMSE       : {final_metrics['RMSE_test']:.4f}")
print(f"    R2         : {final_metrics['R2_test']:.4f}")
print(f"    MAPE       : {final_metrics['MAPE_test']:.2f}%" if final_metrics['MAPE_test'] else "    MAPE       : N/A")
print(f"    sMAPE      : {final_metrics['sMAPE_test']:.2f}%")
print(f"\n  CV Metrics ({N_CV_SPLITS} folds):")
print(f"    MAE        : {final_metrics['CV_MAE_mean']:.4f} (+/- {final_metrics['CV_MAE_std']:.4f})")
print(f"    R2         : {final_metrics['CV_R2_mean']:.4f} (+/- {final_metrics['CV_R2_std']:.4f})")
print(f"\n  Overfit ratio: {final_metrics['Overfit_ratio']:.2f}")
print("="*60)
print("\n[DONE] XGBoost/LightGBM modeling complete!")